# 01 - Full Pipeline Review

## Purpose

Review the final state of the capstone project before optimization.

This notebook inventories the project outputs across Bronze, Silver, Gold, Night Shift, and views.

## Main checks

- Bronze tables exist
- Silver tables exist
- Gold tables exist
- Night Shift outputs exist
- views exist
- row counts are available
- outputs are ready for presentation

## Main idea

Before improving the project, I need a clear view of what already exists and what the final pipeline produces.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

catalog = "vattenfall_dev"

pipeline_objects = [
    # Bronze
    {
        "layer": "Bronze",
        "object_type": "table",
        "object_name": f"{catalog}.raw.bronze_market_prices",
        "purpose": "Raw market price ingestion output",
    },
    {
        "layer": "Bronze",
        "object_type": "table",
        "object_name": f"{catalog}.raw.bronze_weather",
        "purpose": "Raw weather ingestion output",
    },
    {
        "layer": "Bronze",
        "object_type": "table",
        "object_name": f"{catalog}.raw.bronze_grid_events",
        "purpose": "Raw grid event ingestion output",
    },
    {
        "layer": "Bronze",
        "object_type": "table",
        "object_name": f"{catalog}.raw.bronze_asset_reference",
        "purpose": "Raw asset reference ingestion output",
    },

    # Silver
    {
        "layer": "Silver",
        "object_type": "table",
        "object_name": f"{catalog}.refined.silver_market_prices",
        "purpose": "Cleaned market price records",
    },
    {
        "layer": "Silver",
        "object_type": "table",
        "object_name": f"{catalog}.refined.silver_weather",
        "purpose": "Cleaned weather observations",
    },
    {
        "layer": "Silver",
        "object_type": "table",
        "object_name": f"{catalog}.refined.silver_grid_events",
        "purpose": "Cleaned grid event records",
    },
    {
        "layer": "Silver",
        "object_type": "table",
        "object_name": f"{catalog}.refined.silver_asset_reference",
        "purpose": "Cleaned asset reference lookup",
    },
    {
        "layer": "Silver",
        "object_type": "table",
        "object_name": f"{catalog}.refined.silver_regional_operations_base",
        "purpose": "Integrated Silver base for business logic",
    },

    # Gold
    {
        "layer": "Gold",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_daily_market_summary",
        "purpose": "Daily regional market summary",
    },
    {
        "layer": "Gold",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_weather_impact_summary",
        "purpose": "Daily regional weather impact summary",
    },
    {
        "layer": "Gold",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_grid_incident_summary",
        "purpose": "Grid incident summary by day, region, and severity",
    },
    {
        "layer": "Gold",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_regional_operations_dashboard",
        "purpose": "Dashboard-ready regional operations table",
    },

    # Night Shift
    {
        "layer": "Night Shift",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_data_trust_audit",
        "purpose": "Structured trust audit evidence",
    },
    {
        "layer": "Night Shift",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_asset_incident_intelligence",
        "purpose": "Asset-level incident intelligence",
    },
    {
        "layer": "Night Shift",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_weather_grid_risk_correlation",
        "purpose": "Weather-grid risk correlation output",
    },
    {
        "layer": "Night Shift",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_market_operations_stress",
        "purpose": "Market and operations stress output",
    },
    {
        "layer": "Night Shift",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_pipeline_observability_summary",
        "purpose": "Pipeline observability evidence",
    },
    {
        "layer": "Night Shift",
        "object_type": "table",
        "object_name": f"{catalog}.analytics.gold_executive_energy_risk_dashboard_base",
        "purpose": "Base table for executive risk dashboard view",
    },

    # Views
    {
        "layer": "Views",
        "object_type": "view",
        "object_name": f"{catalog}.analytics.vw_regional_operations_dashboard",
        "purpose": "Analyst-facing regional operations dashboard view",
    },
    {
        "layer": "Views",
        "object_type": "view",
        "object_name": f"{catalog}.analytics.vw_daily_incident_trends",
        "purpose": "Daily incident trend view",
    },
    {
        "layer": "Views",
        "object_type": "view",
        "object_name": f"{catalog}.analytics.vw_market_weather_overview",
        "purpose": "Market and weather overview view",
    },
    {
        "layer": "Views",
        "object_type": "view",
        "object_name": f"{catalog}.analytics.vw_executive_energy_risk_dashboard",
        "purpose": "Executive-facing energy risk dashboard view",
    },
]

In [0]:
inventory_rows = []

for item in pipeline_objects:
    object_name = item["object_name"]

    try:
        df = spark.table(object_name)
        row_count = df.count()
        column_count = len(df.columns)
        status = "AVAILABLE"
    except Exception as error:
        row_count = None
        column_count = None
        status = f"ERROR: {str(error)[:120]}"

    inventory_rows.append(
        Row(
            layer=item["layer"],
            object_type=item["object_type"],
            object_name=object_name,
            purpose=item["purpose"],
            row_count=row_count,
            column_count=column_count,
            status=status,
        )
    )

inventory_df = spark.createDataFrame(inventory_rows)

display(inventory_df)

In [0]:
layer_summary_df = (
    inventory_df
    .groupBy("layer")
    .agg(
        F.count("*").alias("object_count"),
        F.sum(F.when(F.col("status") == "AVAILABLE", 1).otherwise(0)).alias("available_objects"),
        F.sum(F.when(F.col("row_count") > 0, 1).otherwise(0)).alias("objects_with_rows"),
    )
    .orderBy("layer")
)

display(layer_summary_df)

In [0]:
presentation_ready_outputs = [
    f"{catalog}.analytics.gold_regional_operations_dashboard",
    f"{catalog}.analytics.vw_regional_operations_dashboard",
    f"{catalog}.analytics.vw_executive_energy_risk_dashboard",
    f"{catalog}.analytics.gold_data_trust_audit",
    f"{catalog}.analytics.gold_pipeline_observability_summary",
]

presentation_rows = []

for object_name in presentation_ready_outputs:
    df = spark.table(object_name)

    presentation_rows.append(
        Row(
            object_name=object_name,
            row_count=df.count(),
            column_count=len(df.columns),
            presentation_use="Final presentation evidence / dashboard-ready output",
        )
    )

presentation_ready_df = spark.createDataFrame(presentation_rows)

display(presentation_ready_df)

In [0]:
inventory_df.createOrReplaceTempView("final_pipeline_inventory")

print("Final pipeline inventory completed.")
print("Objects reviewed:", inventory_df.count())
print("Presentation-ready outputs reviewed:", presentation_ready_df.count())